# Supplementary Figure -- K=46 edge-trim sensitivity

Edge-trim sensitivity of the spatial MS-AA K=46 held-out top-5 result (removing the first and last 10 TRs).
These panels used to be part of main Figure 6 and now live only here.

It uses the CSV summaries exported by the K=46 edge-trim analysis (revision_07):
- untrimmed vs trimmed held-out top-5 timepoint-identification accuracy (mean +/- SEM over the 20 outer participant splits);
- fraction of above-chance accuracy retained after trimming (rest is shown as not applicable: its untrimmed
  above-chance accuracy is near floor, so the ratio is not interpretable);
- top-5 set overlap (Jaccard) between untrimmed and trimmed selections;
- change in top-5 selection frequency for motifs M0-M4.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDGE_DIR = Path("msaa_edge_trim_sensitivity_K46_top5")

ACC_CSV = EDGE_DIR / "edge_trim_accuracy_summary.csv"
JACCARD_CSV = EDGE_DIR / "edge_trim_top5_jaccard_summary.csv"
FREQ_CSV = EDGE_DIR / "edge_trim_selection_frequency_comparison.csv"
ARCH_CSV = EDGE_DIR / "edge_trim_locked_archetype_single_component_summary.csv"

CONDITIONS = ["intact", "word", "rest"]
COND_TICKS = ["Intact\nnarrative", "Word-\nscrambled", "Rest"]
COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}
import sys
sys.path.insert(0, "src")
from msaa_pipeline.paper_archetypes import paper_archetypes
# M0-M4, in the K=46 fit's own numbering (canonical 28,35,16,39,45; legacy 4,5,7,8,9)
LOCKED, ARCHETYPE_LABELS = paper_archetypes()
LABEL = dict(zip(LOCKED, ARCHETYPE_LABELS))

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Supplement_edge_trim_diagnostics.png"
FIG_PDF = OUTPUT_DIR / "Supplement_edge_trim_diagnostics.pdf"

for p in [ACC_CSV, JACCARD_CSV, FREQ_CSV, ARCH_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run analysis/05_robustness/revision_07_K46_edge_trim_heldout_top5_sensitivity.ipynb first."
        )

acc = pd.read_csv(ACC_CSV)
jacc = pd.read_csv(JACCARD_CSV)
freq = pd.read_csv(FREQ_CSV)
arch = pd.read_csv(ARCH_CSV)

In [ ]:
import matplotlib.colorbar as _mcb
_mcb.Colorbar.n_rasterize = 10**9  # keep colorbars vector in PDFs
_add_solids = _mcb.Colorbar._add_solids


def _add_solids_no_seams(self, *a, **k):
    # stroke each colour band in its own colour: no hairline gaps in PDF viewers (matplotlib colorbar docs)
    _add_solids(self, *a, **k)
    if getattr(self, "solids", None) is not None:
        self.solids.set_edgecolor("face")
        self.solids.set_linewidth(0.3)


_mcb.Colorbar._add_solids = _add_solids_no_seams


def vheat(ax, arr, origin="upper", **kw):
    """imshow look-alike drawn as vector cells (pcolormesh), so the PDF holds no raster image."""
    a = np.ma.masked_invalid(np.asarray(arr, dtype=float))
    n, m = a.shape
    im = ax.pcolormesh(np.arange(m + 1) - 0.5, np.arange(n + 1) - 0.5, a, shading="flat",
                       edgecolors="face", linewidth=0.3, **kw)  # face-coloured edges: no seams
    ax.set_xlim(-0.5, m - 0.5)
    ax.set_ylim((-0.5, n - 0.5) if origin == "lower" else (n - 0.5, -0.5))
    return im

# Drawn at the journal's print width (Network Neuroscience reproduces figures ~5 in across), so the point sizes
# are the printed sizes.
plt.rcParams.update({"font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5,
                     "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5,
                     "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
                     "axes.titlepad": 3, "axes.labelpad": 1.5})
fig, axes = plt.subplots(2, 2, figsize=(5.0, 4.2), constrained_layout=True)
fig.get_layout_engine().set(w_pad=0.03, h_pad=0.03, wspace=0.08, hspace=0.08)
axA, axB, axC, axD = axes.ravel()


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9", lw=0.5)
    ax.set_axisbelow(True)


plot_acc = acc.set_index("condition").reindex(CONDITIONS)
x = np.arange(len(CONDITIONS))

# A - accuracy, untrimmed vs trimmed (mean +- SEM over the 20 outer participant splits); conditions are
# categories, so the points are not joined
offset = 0.12
axA.errorbar(x - offset, plot_acc["baseline_accuracy"], yerr=plot_acc["baseline_accuracy_sem"], marker="o",
             linestyle="none", ms=3.5, capsize=1.5, elinewidth=0.8, color="0.25", label="Untrimmed")
axA.errorbar(x + offset, plot_acc["trim_accuracy"], yerr=plot_acc["trim_accuracy_sem"], marker="s",
             linestyle="none", ms=3.5, capsize=1.5, elinewidth=0.8, color="#E4572E", label="First/last 10 TRs trimmed")
axA.axhline(1 / 300, color="0.5", ls="--", lw=0.6)
axA.set_xticks(x)
axA.set_xticklabels(COND_TICKS)
axA.set_xlim(-0.5, len(CONDITIONS) - 0.5)
axA.set_ylabel("Held-out top-5\ntimepoint-identification accuracy")
axA.set_title("Accuracy after edge exclusion", fontweight="bold", loc="left")
axA.set_ylim(0, 1.45 * float(np.nanmax(plot_acc[["baseline_accuracy", "trim_accuracy"]].to_numpy())))  # headroom for the legend
axA.plot([], [], color="0.5", ls="--", lw=0.6, label="Chance (1/300)")
axA.legend(frameon=False, fontsize=6.5, handletextpad=0.3, borderaxespad=0.1, loc="upper right", labelspacing=0.25)
style(axA)

# B - fraction of above-chance accuracy retained; rest is not applicable (near-floor denominator)
frac = plot_acc["fraction_preserved"].to_numpy(dtype=float)
for i, c in enumerate(CONDITIONS):
    if c == "rest":
        # no bar: a plain label, so nothing reads as a value
        axB.text(i, 0.08, "n/a\n(near-floor\ndenominator)", ha="center", va="bottom", fontsize=6.5, color="0.35")
    else:
        axB.bar(i, frac[i], color=COLORS[c])
        axB.text(i, frac[i] + 0.02, f"{frac[i]:.2f}", ha="center", va="bottom", fontsize=6.5)
axB.axhline(1.0, color="0.5", linestyle=":", linewidth=0.7)
axB.set_ylim(0, 1.1)
axB.set_xlim(-0.6, len(CONDITIONS) - 0.4)  # keep the empty rest slot
axB.set_xticks(x)
axB.set_xticklabels(COND_TICKS)
axB.set_ylabel("Fraction of above-chance\naccuracy retained")
axB.set_title("Above-chance accuracy retained", fontweight="bold", loc="left")
style(axB)

# C - top-5 set overlap between untrimmed and trimmed selections
pj = jacc.set_index("condition").reindex(CONDITIONS)
axC.bar(x, pj["mean_jaccard"], color=[COLORS[c] for c in CONDITIONS])
axC.set_ylim(0, 1)
axC.set_xticks(x)
axC.set_xticklabels(COND_TICKS)
axC.set_ylabel("Mean Jaccard overlap")
axC.set_title("Top-5 set overlap,\nuntrimmed vs trimmed", fontweight="bold", loc="left")
style(axC)

# D - M0-M4: trimmed - untrimmed selection frequency
lf = freq[freq["archetype"].isin(LOCKED)].copy()
lf["delta_frequency"] = lf["trim_frequency"] - lf["baseline_frequency"]
heat = (lf.pivot_table(index="archetype", columns="condition", values="delta_frequency", aggfunc="mean")
        .reindex(index=LOCKED, columns=CONDITIONS))
arr = heat.to_numpy(dtype=float)
v = np.nanmax(np.abs(arr))
if not np.isfinite(v) or v == 0:
    v = 1.0
im = vheat(axD, arr, cmap="RdBu_r", vmin=-v, vmax=v)
axD.set_xticks(np.arange(len(CONDITIONS)))
axD.set_xticklabels(COND_TICKS)
axD.set_yticks(np.arange(len(LOCKED)))
axD.set_yticklabels([LABEL[k] for k in LOCKED])
axD.set_title("Change in top-5\nselection frequency", fontweight="bold", loc="left")
axD.tick_params(length=0)
for r in range(arr.shape[0]):
    for c in range(arr.shape[1]):
        if np.isfinite(arr[r, c]):
            axD.text(c, r, f"{arr[r, c]:+.2f}", ha="center", va="center", fontsize=6.5,
                     color="white" if abs(arr[r, c]) > 0.6 * v else "#222222")
cb = fig.colorbar(im, ax=axD, fraction=0.08, pad=0.03)
cb.set_label("Trimmed \u2212 untrimmed", labelpad=2)
cb.ax.tick_params(labelsize=6.5, length=2, width=0.6)
cb.outline.set_linewidth(0.6)

for ax, letter in zip(axes.ravel(), "ABCD"):
    # panel letter leads the title, so it cannot collide with the y-axis label
    ax.set_title(f"{letter}   {ax.get_title(loc='left')}", fontweight="bold", loc="left")

fig.suptitle("Edge-trim sensitivity of the held-out top-5 result (spatial MS-AA, K = 46)",
             fontsize=8.5, fontweight="bold")

fig.savefig(FIG_PNG, dpi=600, bbox_inches="tight", pad_inches=0.02)
fig.savefig(FIG_PDF, dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)